# DSE 200 — Day 4 Lab, part 2
## Three Ways To Get A Model

**Sections 9.1–9.5 match the slides · same weather data, same question, three routes · agent on.**

**Use a GPU runtime for 9.3** (*Runtime → Change runtime type → T4 GPU*). It works on CPU too, just slower.

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while we talk. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost.

In [ ]:
# Installs take 2-3 minutes.
# tabpfn is pinned to 2.0.9: newer versions need a Prior Labs account to download the model.
# autogluon.tabular[all] would conflict with it, so we ask for the four model families we use.
!uv pip install --system -q "autogluon.tabular[lightgbm,catboost,xgboost,fastai]" "tabpfn==2.0.9"

In [ ]:
# ▶ SETUP — run first. Rebuilds the Gradient Boosting notebook's data (about a minute).
import os, time, textwrap, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
        print("downloaded", name, "from the course repo")
        return name
    except Exception as e:
        raise FileNotFoundError(f"{name} not found -- drag it into the Files pane (folder icon, left)") from e

df = pd.read_csv(find("minute_weather.csv.gz"), parse_dates=["hpwren_timestamp"])
df = df.drop_duplicates(subset=df.columns.drop("rowID"))            # the stuck logger (8.3)
df = df.sort_values("hpwren_timestamp").reset_index(drop=True)
df["dry_now"] = (df["relative_humidity"] < 25).astype(int)
df["dry_3h"]  = (df["relative_humidity"].shift(-180) < 25).astype(int)   # the label (8.4)
df = df.iloc[:-180].copy()

# features that look only backward (8.7)
df["hour"]  = df["hpwren_timestamp"].dt.hour
df["month"] = df["hpwren_timestamp"].dt.month
df["rh_change_1h"] = df["relative_humidity"] - df["relative_humidity"].shift(60)
df["rh_change_3h"] = df["relative_humidity"] - df["relative_humidity"].shift(180)
df["temp_change_1h"] = df["air_temp"] - df["air_temp"].shift(60)
df["pressure_change_3h"] = df["air_pressure"] - df["air_pressure"].shift(180)
sensors = ["air_pressure", "air_temp", "avg_wind_direction", "avg_wind_speed",
           "max_wind_direction", "max_wind_speed", "min_wind_direction",
           "min_wind_speed", "rain_accumulation", "rain_duration", "relative_humidity"]
features = sensors + ["hour", "month", "rh_change_1h", "rh_change_3h",
                      "temp_change_1h", "pressure_change_3h"]

# year 1 trains, year 2 chooses settings, year 3 is the final test (8.5)
year1 = df[df["hpwren_timestamp"] < "2012-09-10"]
year2 = df[(df["hpwren_timestamp"] >= "2012-09-10") & (df["hpwren_timestamp"] < "2013-09-10")]
year3 = df[df["hpwren_timestamp"] >= "2013-09-10"]


_READ = {
"9.4": ("XGBoost 0.913, AutoGluon 0.918, TabPFN 0.912 (from 2,000 rows).",
        "On the same 5,000 year-3 minutes. The spread is small: three very different routes\n"
        "land within a couple of points of each other on well-prepared tabular data.\n"
        "TabPFN saw 2,000 rows, 0.4% of year 1, and trained nothing.\n"
        "The model was never the hard part. The features, the split and the baseline were."),
"9.5": ("XGBoost 0.907 against a ceiling of 0.908: about 0.001 below.",
        "On humidity and hour, nothing can beat the ceiling: in every cell, minutes that look\n"
        "identical went both ways. XGBoost lands right under it, and so does a plain table of counts.\n"
        "A better model cannot move the ceiling. Only better columns can."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print(f"{len(df):,} minutes. Ready.")

# The same 5,000 year-3 minutes score all three routes (TabPFN is too slow to score all 500,000 on a CPU)
test = year3.sample(5000, random_state=42)
from sklearn.metrics import average_precision_score
results = {}
def record(name, prob, seconds):
    pred = (prob >= 0.5).astype(int)
    results[name] = {"accuracy": (pred == test["dry_3h"]).mean(),
                     "PR-AUC": average_precision_score(test["dry_3h"], prob),
                     "seconds": round(seconds)}
    print(name, {k: round(v, 3) for k, v in results[name].items()})
print("same as now on these minutes:", round((test["dry_now"] == test["dry_3h"]).mean(), 3))

# One: You Tune It

*Slides: Three Ways To Get A Model · section 9.1*

## 9.1 XGBoost

What you did this morning. You pick the knobs, and you know why the model is good because you watched it get there.

In [ ]:
# ▶ DEMO
start = time.time()
xgb = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                    early_stopping_rounds=20, random_state=42)
xgb.fit(year1[features], year1["dry_3h"], eval_set=[(year2[features], year2["dry_3h"])], verbose=False)
record("XGBoost, tuned by you", xgb.predict_proba(test[features])[:, 1], time.time() - start)

# Two: It Searches For You

*Slides: Three Ways To Get A Model · section 9.2*

## 9.2 AutoGluon

One call. A time budget in seconds. A ranked table at the end. It fits many models (trees, boosted ensembles, neural nets, nearest neighbours) and then **stacks** them: the predictions of every model become features for one final model.

We hand it year 2 as `tuning_data`, the same job year 2 did for early stopping. Otherwise it would cut its own random validation rows, and you saw in 8.5 what a random split does to minutes.

In [ ]:
# ▶ DEMO -- ten minutes. Start it, then read ahead.
from autogluon.tabular import TabularPredictor

cols = features + ["dry_3h"]
start = time.time()
predictor = TabularPredictor(label="dry_3h", verbosity=1).fit(
    year1[cols], tuning_data=year2[cols], time_limit=600)
record("AutoGluon, 10 min budget", predictor.predict_proba(test[features])[1].values, time.time() - start)

In [ ]:
# ▶ DEMO -- what it built, ranked on our test minutes
predictor.leaderboard(test[cols])[["model", "score_test", "score_val", "fit_time"]].head(8)

# Three: It Does Not Train

*Slides: Three Ways To Get A Model · section 9.3*

## 9.3 TabPFN

`fit` does not fit anything. It **remembers** your rows. A transformer was pre-trained once on millions of **synthetic** tables; at prediction time your training rows and test rows go in together, and it answers by pattern, the way an LLM reads the text above the cursor. That is in-context learning, on a table.

Your rows live in its context window, so it cannot take 500,000 of them. We **subsample** 2,000.

In [ ]:
# ▶ DEMO -- 2,000 rows of year 1. On a GPU: seconds. On a CPU: a few minutes (prediction is the slow part).
from tabpfn import TabPFNClassifier

small = year1.sample(2000, random_state=42)
start = time.time()
pfn = TabPFNClassifier(random_state=42, ignore_pretraining_limits=True)
pfn.fit(small[features], small["dry_3h"])
print("fit took", round(time.time() - start, 1), "seconds")
record("TabPFN, 2,000-row sample", pfn.predict_proba(test[features])[:, 1], time.time() - start)

# Before We Look

*Slides: Three Ways To Get A Model · section 9.4*

## 9.4 Write Down Your Guess

In [ ]:
# 🔍 READ 9.4 -- accuracy on the same 5,000 year-3 minutes. Fill in your guesses, then run.
#
#     XGBoost, tuned by you        ?
#     AutoGluon, 10 min budget     ?
#     TabPFN, 2,000-row sample     ?

prediction = "???"
check_read("9.4", prediction)

In [ ]:
# ▶ DEMO -- the table
pd.DataFrame(results).T.round(3)

**How much accuracy did the subsample cost?** Same 5,000 test minutes; TabPFN and a plain XGBoost each get the same small sample of year 1.

In [ ]:
# ▶ DEMO -- a few minutes on CPU
for n in [250, 500, 1000]:
    s = year1.sample(n, random_state=42)
    p1 = TabPFNClassifier(random_state=42).fit(s[features], s["dry_3h"]).predict(test[features])
    p2 = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3).fit(s[features], s["dry_3h"]).predict(test[features])
    print(f"{n:>5} rows   TabPFN {(p1 == test['dry_3h']).mean():.3f}   XGBoost {(p2 == test['dry_3h']).mean():.3f}")

## ✍️ Your Turn: One Table, One Paragraph

The table above has accuracy, PR-AUC (the metric that suits a 26/74 split: it ignores the easy "not dry" minutes) and wall-clock time.

**Which would you ship, and to whom would you have to justify it?** Choose by constraint, not by leaderboard:

| If you need | Reach for |
| --- | --- |
| to explain every decision | the model you tuned |
| the best number, and can spend compute | AutoML |
| an answer in seconds on small data | a tabular foundation model |
| to know whether the problem is learnable at all | any of them, once |

You will be asked why TabPFN needed a subsample. Have an answer.

*Your paragraph:*

# Why They All Land Together

*Slides: Three Ways To Get A Model · section 9.5*

## 9.5 Bayes And The Ceiling

Every classifier is estimating one thing: **P(dry | what it can see)**. Bayes' rule says P(dry | x) = P(x | dry) · P(dry) / P(x). With enough rows that is just counting.

We use two coarse columns so we *can* count: humidity in bins of 5%, and the hour.

In [ ]:
# ▶ DEMO -- Bayes' rule, by counting
def add_bin(d):
    return d.assign(rh_bin=(d["relative_humidity"] // 5 * 5).clip(upper=95))
y1, y2, y3 = add_bin(year1), add_bin(year2), add_bin(year3)

cell = y1["rh_bin"] == 30                          # humidity 30-35% now
p_dry         = y1["dry_3h"].mean()
p_cell_if_dry = cell[y1["dry_3h"] == 1].mean()
p_cell        = cell.mean()
print("Bayes' rule:", round(p_cell_if_dry * p_dry / p_cell, 3))
print("just count :", round(y1.loc[cell, "dry_3h"].mean(), 3))

y1.groupby("rh_bin")["dry_3h"].mean().plot.bar(figsize=(9, 4))
plt.axhline(0.5, color="k", ls="--"); plt.ylabel("P(dry in 3 hours)"); plt.show()

**The best possible rule** (the *Bayes optimal classifier*): in every cell, say the more likely answer. Nothing that sees only these columns can beat it, because minutes that look identical went both ways:

In [ ]:
# ▶ DEMO -- minutes that look the same, with different futures
twins = y3[(y3["rh_bin"] == 30) & (y3["hour"] == 14) & (y3["rh_change_3h"] < 0)]
print(len(twins), "minutes look identical;", twins["dry_3h"].sum(), "were dry 3 hours later.")

In [ ]:
# 🔍 READ 9.5 -- XGBoost on ONLY rh_bin and hour, scored on all of year 3.
#
#     Below you compute the best score ANY rule on those two columns could get.
#     How far below that ceiling does XGBoost land? Commit to a number.

prediction = "???"
check_read("9.5", prediction)

In [ ]:
# ✍️ YOU TRY 9.5
cells = ["rh_bin", "hour"]
g = y3.groupby(cells)["dry_3h"].agg(["sum", "count"])    # dry minutes, all minutes, per cell

ceiling = None   # TODO: the majority answer is right max(sum, count - sum) times in each cell.
                 #       Add those up over all cells and divide by len(y3).
two = None       # TODO: XGBoost on y1[cells] (early stopping on y2), then .score on y3

In [ ]:
# ✅ CHECKPOINT 9.5
assert ceiling is not None and two is not None, "compute the ceiling and the XGBoost score"
print("ceiling", round(ceiling, 3), "  XGBoost", round(two, 3))
assert two <= ceiling + 1e-9, "nothing on these columns can beat the ceiling: check your ceiling"
assert ceiling - two < 0.01, "XGBoost should land just under the ceiling: check the fit"
print("checkpoint passed: the model hits the ceiling. Only new columns can raise it.")

In [ ]:
# ▶ DEMO -- why we need models at all: counting runs out
for cols in [["rh_bin"], ["rh_bin", "hour"], features]:
    print(f"{len(cols):>2} columns: {len(y3) / y3.groupby(cols, dropna=False).ngroups:>9,.0f} minutes per cell")

With all 17 columns every minute is alone in its cell, so there is nothing to count. A model **estimates** P(y | x) where counting cannot. XGBoost, AutoGluon and TabPFN are three ways of estimating the same thing, pressing on the same ceiling. That is why they land together.

**Seeing is not doing.** Even a perfect P(y | x) answers "among minutes like this, how often?" Remember the kidney stones from Causal Thinking: a perfect classifier on treatment alone says keyhole patients clear more often (83% vs 78%), and as a prediction that is right. As a decision it is wrong: open surgery wins for small stones and for large. No leaderboard measures that.

### ✍️ In one sentence
**A vendor says their AutoML will take this model from 0.91 to 0.97 on the same columns. What do you ask them?**

*Your answer:*